# Getting started with ztforce

This notebook walks through the core workflow: running forced PSF photometry at a fixed sky position, inspecting the resulting lightcurve, and saving it to disk.

**Prerequisites:** install ztforce and set your IRSA credentials (see the [installation docs](https://ztforce.readthedocs.io)).

```bash
pip install ztforce
export ZTFORCE_IRSA_USER=you@example.com
export ZTFORCE_IRSA_PASS=secret
```

## Run forced photometry

`run_forced_photometry` is the main entry point. Pass an RA/Dec (decimal degrees, J2000) and a list of bands. It returns a dict mapping band → `Lightcurve`.

In [ ]:
from ztforce import run_forced_photometry

# Bright star near Praesepe (M44) — PS1 g = 15.82
lcs = run_forced_photometry(
    ra=130.13113,
    dec=19.69525,
    bands=["g", "r"],
    max_epochs=50,  # cap to the 50 most recent epochs for a quick demo
)
print(lcs)

## Inspect the lightcurve DataFrame

Each `Lightcurve` exposes its data as a pandas DataFrame via `.df`. The `detection` column flags epochs with SNR ≥ 3; `upper_limit` gives the 5σ limiting magnitude for non-detections.

In [ ]:
lc_g = lcs["g"]
df = lc_g.df
print(df.columns.tolist())
df[["obsjd", "mag", "mag_err", "snr", "detection", "flags"]].head(10)

## Plot

`.plot()` renders detections as points with error bars and non-detections as upper-limit arrows.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
lcs["g"].plot(ax=axes[0])
lcs["r"].plot(ax=axes[1])
axes[0].set_title("ZTF g-band")
axes[1].set_title("ZTF r-band")
plt.tight_layout()
plt.show()

## Stack detections

`.stack()` computes an inverse-variance-weighted (IVW) combination of all clean detections, returning a DataFrame indexed by band.

In [ ]:
lc_g.stack()

You can restrict the stack to a JD window, e.g. to compare pre- and post-outburst brightness:

```python
lc_g.stack(jd_min=2460000, jd_max=2460200)
```

For a sliding window stack use `.rolling_stack(window=30)` (window in days).

## Save and reload

Lightcurves are serialised as Astropy ECSV files, which preserve column types and source coordinates.

In [ ]:
from ztforce import Lightcurve

lc_g.save("my_source_g.ecsv")

lc_reloaded = Lightcurve.load("my_source_g.ecsv")
print(lc_reloaded)

## Batch mode

To run on many targets at once, pass a list of `SkyCoord` objects to `run_forced_photometry_batch`:

```python
from astropy.coordinates import SkyCoord
from ztforce import run_forced_photometry_batch

targets = SkyCoord(ra=[130.13, 210.08], dec=[19.70, -6.88], unit="deg")
results = run_forced_photometry_batch(targets, bands=["g", "r"])
# results[0]["g"] -> Lightcurve for targets[0]
```

FITS cutouts are shared automatically when multiple targets fall on the same image.